# 04 — PointNet++ baselines on 3DSES Gold (Phase 2: T2.1–T2.4)

PointNet++ (OpenPoints, `cfgs/s3dis/pointnet++.yaml`), inputs XYZ + RGB + height (+ intensity), 2 cm data, 18 classes.
Train on S164, S165, S168, S178, S179 · validate on **S163 + S169** (team decision 2026-11-12).
Headline small-component classes (fixed 2026-11-12): Damper, Exit sign, Fire terminal, Outlet, Switch.

- **B0** = `use_intensity=False` (EXP-001, seed 0) · **B1** = `use_intensity=True` (EXP-002, seed 0).
- No attention, no special sampling, no class weighting: these are the numbers everything else is compared with.
- For a new run change only `run_name`, `use_intensity` and `seed` in the first code cell. **One Kaggle version = one run.**

**Since 2026-11-13 the run finishes the whole evaluation by itself** (a checkpoint cannot be fetched from an older Kaggle version later, see EVAL-001):
1. validation on the sub-sampled points (as before);
2. **validation on every original point** of S163 + S169, the oracle ceiling of the voxel size, and where the points of the small classes go;
3. **labels for every point of the test stations** S170, S171, S180 → `pred_test/S17xlabels.npy` (the files a Codabench submission is made of).

**How to run:** Settings → Accelerator **GPU T4 x2**, then Save Version → *Save & Run All* (Advanced: "Run with GPU for this session").
On CPU (the automatic run after import) it only does a smoke test of the data pipeline, the training code and all evaluation steps with a tiny stand-in model.

The run checkpoints every epoch, stops by itself before the time budget, and writes everything to `/kaggle/working/<run_name>/`.

In [ ]:
import os, sys, json, time, glob, math, subprocess
import numpy as np

CFG = dict(
    run_name="EXP-006_B0_pointnet2_rgb_2cm_seed2",
    voxel="2cm",                 # which pre-processed resolution: "2cm" or "1cm"
    use_intensity=False,         # B0: False, B1: True
    npts=24000,                  # points per training sample (as in the OpenPoints S3DIS setup)
    batch=8,
    iters_per_epoch=300,
    epochs=40,
    lr=0.01, min_lr=1e-5, weight_decay=1e-4, label_smoothing=0.2,
    val_every=2,                 # validate every N epochs
    time_budget_h=7.5,           # stop training after this many hours (Kaggle sessions end at 12 h)
    seed=2,                      # EXP-001 / EXP-002 used seed 0
    train=["S164", "S165", "S168", "S178", "S179"],
    val=["S163", "S169"],
    small=[3, 5, 6, 10, 14],     # Damper, Exit sign, Fire terminal, Outlet, Switch
    num_classes=18,
    fullres_val=True,            # after training: score the best checkpoint on every original validation point
    predict_test=True,           # after training: write one label per original point of the test stations
    test=["S170", "S171", "S180"],
    focus=[1, 3, 5, 6, 10, 14],  # classes whose confusions are printed: Components + the five small ones
)
CLASS_NAMES = ["Column", "Components", "Covering", "Damper", "Door", "Exit sign", "Fire terminal", "Furniture",
               "Heater", "Lamp", "Outlet", "Railing", "Slab", "Stair", "Switch", "Wall", "Window", "Clutter"]
VOXEL_M = {"2cm": 0.02, "1cm": 0.01}[CFG["voxel"]]
OFFSET = np.array([900.0, 480.0, 100.0])
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.path.join("/kaggle/working" if ON_KAGGLE else "./results", CFG["run_name"])
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(CFG["seed"])
print(json.dumps(CFG, indent=1)); print("OUT:", OUT)

## Data

Uses the output of notebook `03-preprocess-gold` if it is attached as input; otherwise it rebuilds the same files itself (download + voxel subsampling, about two minutes).

In [ ]:
STATIONS = CFG["train"] + CFG["val"]

def find_preprocessed():
    for root in [os.environ.get("DATA_DIR", ""), "/kaggle/input", "/kaggle/temp/3dses_gold_v1", "./results/3dses_gold_v1"]:
        if not root or not os.path.isdir(root): continue
        hits = glob.glob(os.path.join(root, "**", CFG["voxel"], STATIONS[0] + ".npz"), recursive=True)
        if hits and all(os.path.isfile(os.path.join(os.path.dirname(hits[0]), s + ".npz")) for s in STATIONS):
            return os.path.dirname(hits[0])
    return None

def voxel_subsample(xyz, size):
    g = np.floor(xyz / size).astype(np.int64)
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)
    order = np.lexsort((d2, key)); ks = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = ks[1:] != ks[:-1]
    return order[first]

def rebuild(out_dir):
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    raw = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=["Gold/*"], local_dir=raw)
    os.makedirs(out_dir, exist_ok=True)
    for s in STATIONS:
        a = np.load(os.path.join(raw, "Gold", s + ".npy")); xyz = a[:, :3] - OFFSET
        keep = voxel_subsample(xyz, VOXEL_M)
        np.savez(os.path.join(out_dir, s + ".npz"), xyz=xyz[keep].astype(np.float32), rgb=np.clip(a[keep, 3:6], 0, 255).astype(np.uint8),
                 intensity=a[keep, 6].astype(np.float32), label=a[keep, 7].astype(np.uint8))
        print("rebuilt", s, len(keep))
    return out_dir

DATA = find_preprocessed()
print("pre-processed data found at:" if DATA else "pre-processed data not attached -> rebuilding", DATA or "")
if DATA is None:
    DATA = rebuild(os.path.join("/kaggle/temp/3dses_gold_v1" if ON_KAGGLE else "./results/3dses_gold_v1", CFG["voxel"]))

from scipy.spatial import cKDTree
D = {}
for s in STATIONS:
    d = np.load(os.path.join(DATA, s + ".npz"))
    D[s] = dict(xyz=d["xyz"].astype(np.float32), rgb=d["rgb"], intensity=d["intensity"].astype(np.float32),
                label=d["label"].astype(np.int64))
    D[s]["tree"] = cKDTree(D[s]["xyz"])
    print(f"{s}: {len(D[s]['xyz']):>8,} points, classes present: {sorted(np.unique(D[s]['label']).tolist())}")
n_train = sum(len(D[s]["xyz"]) for s in CFG["train"]); n_val = sum(len(D[s]["xyz"]) for s in CFG["val"])
print(f"train points: {n_train:,} | val points: {n_val:,}")
IN_CH = 4 + int(CFG["use_intensity"])       # RGB + height (+ intensity)

## Sampling, features, metrics (NumPy only)

In [ ]:
def crop_indices(st, center_idx, k):
    """Indices of the k points nearest to one point of the station (the OpenPoints 'voxel_max' crop)."""
    _, idx = st["tree"].query(st["xyz"][center_idx], k=min(k, len(st["xyz"])))
    return np.asarray(idx)

def make_sample(st, idx, train, rng):
    """Returns pos [n,3] float32, feat [C,n] float32, label [n] int64 for one crop."""
    xyz = st["xyz"][idx].astype(np.float32).copy()
    rgb = st["rgb"][idx].astype(np.float32) / 127.5 - 1.0                     # [-1, 1]
    if train:
        a = rng.uniform(0, 2 * math.pi); c, s_ = math.cos(a), math.sin(a)      # rotation around the vertical axis
        ctr = xyz[:, :2].mean(0)
        xy = xyz[:, :2] - ctr
        xyz[:, :2] = np.stack([c * xy[:, 0] - s_ * xy[:, 1], s_ * xy[:, 0] + c * xy[:, 1]], 1) + ctr
        xyz *= rng.uniform(0.9, 1.1)                                           # scaling
        xyz += np.clip(rng.normal(0, 0.005, xyz.shape), -0.02, 0.02).astype(np.float32)   # jitter
        if rng.random() < 0.2: rgb[:] = 0.0                                    # colour drop
    pos = xyz - np.array([xyz[:, 0].mean(), xyz[:, 1].mean(), xyz[:, 2].min()], dtype=np.float32)
    feats = [rgb, pos[:, 2:3]]                                                 # RGB + height above the lowest point of the crop
    if CFG["use_intensity"]: feats.append((st["intensity"][idx] * 2.0 - 1.0)[:, None])
    return pos.astype(np.float32), np.concatenate(feats, 1).T.astype(np.float32).copy(), st["label"][idx]

_train_p = np.array([len(D[s]["xyz"]) for s in CFG["train"]], dtype=np.float64); _train_p /= _train_p.sum()

def train_batch_np(rng):
    P, X, Y = [], [], []
    for _ in range(CFG["batch"]):
        st = D[CFG["train"][rng.choice(len(CFG["train"]), p=_train_p)]]       # station chosen by size ...
        idx = crop_indices(st, rng.integers(len(st["xyz"])), CFG["npts"])      # ... then a uniformly random centre point
        p, x, y = make_sample(st, idx, True, rng); P.append(p); X.append(x); Y.append(y)
    return np.stack(P), np.stack(X), np.stack(Y)

def confusion(gt, pred, C):
    return np.bincount(gt * C + pred, minlength=C * C).reshape(C, C)

def metrics_from_confusion(cm, small):
    tp = np.diag(cm).astype(np.float64); gt = cm.sum(1); pr = cm.sum(0)
    present = gt > 0
    iou = np.where(present, tp / np.maximum(gt + pr - tp, 1), np.nan)
    acc = np.where(present, tp / np.maximum(gt, 1), np.nan)
    sm = [c for c in small if present[c]]
    return dict(mIoU=float(np.nanmean(iou) * 100), mAcc=float(np.nanmean(acc) * 100), OA=float(tp.sum() / max(cm.sum(), 1) * 100),
                small_mIoU=float(np.mean([iou[c] for c in sm]) * 100) if sm else float("nan"),
                iou=[None if np.isnan(v) else round(float(v) * 100, 2) for v in iou], n_classes_present=int(present.sum()))

def coverage_crops(st, k, rng, group=4):
    """Yields groups of crops (index arrays) until every point of the station has been in at least one crop."""
    N = len(st["xyz"]); covered = np.zeros(N, dtype=bool)
    while not covered.all():
        todo = np.flatnonzero(~covered)
        centres = rng.choice(todo, size=min(group, len(todo)), replace=False)
        crops = [crop_indices(st, c, k) for c in centres]
        for idx in crops: covered[idx] = True
        yield crops

# quick self-checks
_p, _x, _y = train_batch_np(np.random.default_rng(1))
assert _p.shape == (CFG["batch"], CFG["npts"], 3) and _x.shape == (CFG["batch"], IN_CH, CFG["npts"]) and _y.shape == (CFG["batch"], CFG["npts"])
assert np.isfinite(_p).all() and np.isfinite(_x).all() and _y.min() >= 0 and _y.max() < CFG["num_classes"]
_cm = confusion(np.array([0, 1, 1, 2]), np.array([0, 1, 2, 2]), 3); _m = metrics_from_confusion(_cm, [1])
assert abs(_m["OA"] - 75) < 1e-6 and abs(_m["iou"][1] - 50) < 1e-6
_n = sum(len(g) for g in coverage_crops(D[CFG["val"][0]], CFG["npts"], np.random.default_rng(0)))
print(f"batch shapes OK {_p.shape} {_x.shape}; crop extent (m): {np.ptp(_p[0], axis=0).round(2)}; "
      f"full coverage of {CFG['val'][0]} takes {_n} crops")

## Model

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(CFG["seed"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = DEV != "cuda"
print("PyTorch", torch.__version__, "| device:", DEV, "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "SMOKE TEST (no GPU): tiny stand-in model, a few steps")

class DummySeg(nn.Module):
    """Stand-in used only for the CPU smoke test (the real model needs CUDA extensions)."""
    def __init__(self, in_ch, n_cls):
        super().__init__(); self.net = nn.Sequential(nn.Conv1d(in_ch + 3, 32, 1), nn.ReLU(), nn.Conv1d(32, n_cls, 1))
    def forward(self, data):
        return self.net(torch.cat([data["pos"].transpose(1, 2), data["x"]], 1))

def sh(cmd, cwd=None, timeout=1500):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return p.returncode == 0, "\n".join((p.stdout + "\n" + p.stderr).strip().splitlines()[-25:])

def build_openpoints_pointnet2(in_ch, n_cls):
    src = "/kaggle/temp/code" if ON_KAGGLE else "./_code"; os.makedirs(src, exist_ok=True)
    repo = os.path.join(src, "PointNeXt")
    sh("git config --global url.https://github.com/.insteadOf git@github.com:")       # submodule is listed with an SSH address
    if not os.path.isdir(os.path.join(repo, "openpoints", "models")):
        ok, out = sh(f"git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/guochengqian/PointNeXt {repo}")
        if not ok: raise RuntimeError("clone failed:\n" + out)
    t0 = time.time()
    ok, out = sh(f"{sys.executable} setup.py install", cwd=os.path.join(repo, "openpoints", "cpp", "pointnet2_batch"))
    if not ok: raise RuntimeError("compiling pointnet2_batch failed:\n" + out)
    print(f"compiled pointnet2_batch in {time.time() - t0:.0f} s")
    sys.path.insert(0, repo)
    for _ in range(25):
        try:
            for k in [k for k in sys.modules if k == "openpoints" or k.startswith("openpoints.")]: del sys.modules[k]
            from openpoints.utils import EasyConfig
            from openpoints.models import build_model_from_cfg
            break
        except ModuleNotFoundError as e:
            pkg = e.name.split(".")[0]
            if pkg.startswith(("openpoints", "pointnet2")): raise
            ok, out = sh(f"{sys.executable} -m pip install -q {pkg}", timeout=600); print("pip install", pkg, ok)
            if not ok: raise RuntimeError(out)
    cfg = EasyConfig(); cfg.load(os.path.join(repo, "cfgs", "s3dis", "pointnet++.yaml"), recursive=True)
    cfg["model"]["encoder_args"]["in_channels"] = in_ch
    cfg["model"]["cls_args"]["num_classes"] = n_cls
    ok, commit = sh("git log -1 --format=%h", cwd=repo)
    return build_model_from_cfg(cfg.model), commit.strip()

if SMOKE:
    model, code_commit = DummySeg(IN_CH, CFG["num_classes"]), "dummy"
else:
    model, code_commit = build_openpoints_pointnet2(IN_CH, CFG["num_classes"])
model = model.to(DEV)
print("model:", type(model).__name__, "| parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "| code commit:", code_commit)

## Training and validation code

In [ ]:
EPOCHS = 1 if SMOKE else CFG["epochs"]
ITERS = 3 if SMOKE else CFG["iters_per_epoch"]
TOTAL = EPOCHS * ITERS
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
def lr_at(it):                                        # cosine schedule from lr to min_lr, no warm-up
    return CFG["min_lr"] + 0.5 * (CFG["lr"] - CFG["min_lr"]) * (1 + math.cos(math.pi * min(it, TOTAL) / TOTAL))

def to_batch(P, X):
    return {"pos": torch.from_numpy(P).to(DEV), "x": torch.from_numpy(X).to(DEV)}

def train_step(it):
    for g in opt.param_groups: g["lr"] = lr_at(it)
    P, X, Y = train_batch_np(rng)
    logits = model(to_batch(P, X))                                             # [B, C, N]
    loss = F.cross_entropy(logits, torch.from_numpy(Y).to(DEV), label_smoothing=CFG["label_smoothing"])
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return float(loss.detach())

@torch.no_grad()
def predict_station(st, max_groups=None):
    """Class scores for every point of a station: overlapping crops, summed probabilities."""
    model.eval()
    N, C = len(st["xyz"]), CFG["num_classes"]
    score = np.zeros((N, C), dtype=np.float32); r = np.random.default_rng(0); n_groups = 0
    for crops in coverage_crops(st, CFG["npts"], r):
        S = [make_sample(st, idx, False, r) for idx in crops]
        prob = F.softmax(model(to_batch(np.stack([s[0] for s in S]), np.stack([s[1] for s in S]))), 1)   # [g, C, n]
        prob = prob.transpose(1, 2).cpu().numpy()
        for idx, p in zip(crops, prob): score[idx] += p
        n_groups += 1
        if max_groups and n_groups >= max_groups: break
    return score

def validate(max_groups=None):
    C = CFG["num_classes"]; cm = np.zeros((C, C), dtype=np.int64)
    for s in CFG["val"]:
        score = predict_station(D[s], max_groups)
        seen = score.sum(1) > 0                                                # (only matters when max_groups cuts the pass short)
        cm += confusion(D[s]["label"][seen], score[seen].argmax(1), C)
    model.train()
    return metrics_from_confusion(cm, CFG["small"]), cm

def save_ckpt(path, epoch, it, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "it": it, "best": best, "cfg": CFG}, path)

def find_resume():
    cands = [os.path.join(OUT, "ckpt_last.pth")] + glob.glob(os.path.join("/kaggle/input", "**", CFG["run_name"], "ckpt_last.pth"), recursive=True)
    return next((c for c in cands if os.path.isfile(c)), None)

# ---- pre-flight: exercise every code path once before the long loop
model.train()
t0 = time.time(); l0 = train_step(0)
m0, _ = validate(max_groups=1)
save_ckpt(os.path.join(OUT, "ckpt_preflight.pth"), 0, 0, -1.0)
model.load_state_dict(torch.load(os.path.join(OUT, "ckpt_preflight.pth"), map_location=DEV)["model"])
os.remove(os.path.join(OUT, "ckpt_preflight.pth"))
print(f"pre-flight OK in {time.time() - t0:.1f} s: loss {l0:.3f}, partial-val mIoU {m0['mIoU']:.2f}")

## Train

In [ ]:
start_epoch, it, best = 0, 0, -1.0
res = find_resume()
if res:
    ck = torch.load(res, map_location=DEV)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start_epoch, it, best = ck["epoch"], ck["it"], ck["best"]
    print(f"resumed from {res}: epoch {start_epoch}, best val mIoU {best:.2f}")

log_path = os.path.join(OUT, "log.jsonl"); T0 = time.time(); stopped_early = False
for epoch in range(start_epoch, EPOCHS):
    model.train(); losses = []; te = time.time()
    for _ in range(ITERS):
        losses.append(train_step(it)); it += 1
    rec = {"epoch": epoch + 1, "it": it, "loss": round(float(np.mean(losses)), 4), "lr": round(lr_at(it), 6),
           "epoch_s": round(time.time() - te, 1), "elapsed_h": round((time.time() - T0) / 3600, 3)}
    if (epoch + 1) % CFG["val_every"] == 0 or epoch + 1 == EPOCHS:
        tv = time.time(); m, cm = validate()
        rec.update(val_mIoU=round(m["mIoU"], 2), val_small_mIoU=round(m["small_mIoU"], 2), val_OA=round(m["OA"], 2), val_s=round(time.time() - tv, 1))
        if m["mIoU"] > best:
            best = m["mIoU"]; save_ckpt(os.path.join(OUT, "ckpt_best.pth"), epoch + 1, it, best); rec["best"] = True
    save_ckpt(os.path.join(OUT, "ckpt_last.pth"), epoch + 1, it, best)
    with open(log_path, "a") as fh: fh.write(json.dumps(rec) + "\n")
    print(rec)
    if (time.time() - T0) / 3600 > CFG["time_budget_h"]:
        stopped_early = True; print("time budget reached - stopping; the run can be resumed from ckpt_last.pth"); break
print(f"training finished: {(time.time() - T0) / 3600:.2f} h, best val mIoU {best:.2f}")

## Final validation of the best checkpoint

In [ ]:
bp = os.path.join(OUT, "ckpt_best.pth")
if os.path.isfile(bp):
    ck = torch.load(bp, map_location=DEV); model.load_state_dict(ck["model"]); best_epoch = ck["epoch"]
else:
    best_epoch = None
PRED = {}; cm = np.zeros((CFG["num_classes"], CFG["num_classes"]), dtype=np.int64)       # same computation as validate(), predictions kept
for s in CFG["val"]:
    PRED[s] = predict_station(D[s]).argmax(1).astype(np.uint8)
    cm += confusion(D[s]["label"], PRED[s].astype(np.int64), CFG["num_classes"])
    np.save(os.path.join(OUT, f"pred_val_{CFG['voxel']}_{s}.npy"), PRED[s])
model.train()
m = metrics_from_confusion(cm, CFG["small"])
np.save(os.path.join(OUT, "confusion_val.npy"), cm)
gt_points = cm.sum(1)
print(f"\n{CFG['run_name']}  |  validation = {'+'.join(CFG['val'])}  |  best epoch {best_epoch}  |  {'SMOKE TEST - numbers are meaningless' if SMOKE else 'GPU run'}")
print(f"mIoU {m['mIoU']:.2f}  |  small-component mIoU {m['small_mIoU']:.2f}  |  mAcc {m['mAcc']:.2f}  |  OA {m['OA']:.2f}  |  classes present in validation: {m['n_classes_present']}")
print(f"\n{'ID':>2} {'class':<14} {'val points':>10} {'IoU %':>7}")
for c in range(CFG["num_classes"]):
    if m["iou"][c] is not None:
        print(f"{c:>2} {CLASS_NAMES[c]:<14} {gt_points[c]:>10,} {m['iou'][c]:>7.2f}{'   <- small component' if c in CFG['small'] else ''}")
final = {"run": CFG["run_name"], "smoke": SMOKE, "cfg": CFG, "code_commit": code_commit, "torch": torch.__version__,
         "gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else None, "best_epoch": best_epoch, "stopped_early": stopped_early,
         "metrics": m, "val_points_per_class": gt_points.tolist(), "class_names": CLASS_NAMES,
         "note": "'metrics' = on the sub-sampled validation points; 'fullres' (added by the next cell) = on every original validation point"}
with open(os.path.join(OUT, "final_metrics.json"), "w") as fh: json.dump(final, fh, indent=1)
print("\nsaved:", sorted(os.listdir(OUT)))

## Validation on every original point

Each original point takes the predicted label of its voxel's representative (the point the network actually saw).
**Oracle** = the score a perfect network would get this way: the ceiling that the voxel size alone puts on each class.
For Components and the five small classes the cell also prints which classes their points are predicted as.

In [ ]:
def voxel_subsample_inv(xyz, size):
    """Same selection as voxel_subsample(), plus inv: original point -> position of its voxel's representative in the sub-sampled cloud."""
    g = np.floor(xyz / size).astype(np.int64)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)
    g -= g.min(0)                                           # a whole-voxel shift: same voxels, keys stay non-negative
    assert g.max() < (1 << 20)
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    order = np.lexsort((d2, key)); ks = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = ks[1:] != ks[:-1]
    inv = np.empty(len(order), dtype=np.int64); inv[order] = np.cumsum(first) - 1
    return order[first], inv

def raw_file(sub, s):
    """Path of an original station file (Gold/S163.npy, test_area/S170.npy); downloaded if it is not there yet."""
    rel = os.path.join(sub, s + ".npy")
    for root in [os.environ.get("RAW_DIR", ""), "/kaggle/temp/3dses", "./data/3dses"]:
        if root and os.path.isfile(os.path.join(root, rel)): return os.path.join(root, rel)
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    raw = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=[sub + "/" + s + ".npy"], local_dir=raw)
    assert os.path.isfile(os.path.join(raw, rel)), "download did not produce " + rel
    return os.path.join(raw, rel)

def readout(cm, c, top=4):
    """Where the points of class c go (recall side) and how clean the predictions of class c are (precision side)."""
    row = cm[c]; n = int(row.sum()); pred_n = int(cm[:, c].sum())
    if n == 0: return f"{CLASS_NAMES[c]:<14} no points"
    parts = ", ".join(f"{CLASS_NAMES[j]} {100 * row[j] / n:.1f}%" for j in np.argsort(-row)[:top] if row[j] > 0)
    prec = f"{100 * cm[c, c] / pred_n:.1f}%" if pred_n else "n/a"
    return f"{CLASS_NAMES[c]:<14} {n:>9,} points -> predicted as: {parts}  ||  predicted as {CLASS_NAMES[c]}: {pred_n:,} points, {prec} correct"

def fullres_validation():
    C = CFG["num_classes"]; cm_full = np.zeros((C, C), dtype=np.int64); cm_orc = np.zeros((C, C), dtype=np.int64); how = {}
    for s in CFG["val"]:
        a = np.load(raw_file("Gold", s)); xyz = a[:, :3] - OFFSET; lab = a[:, 7].astype(np.int64)
        assert lab.min() >= 0 and lab.max() < C
        keep, inv = voxel_subsample_inv(xyz, VOXEL_M)
        if len(keep) == len(D[s]["xyz"]) and np.array_equal(xyz[keep].astype(np.float32), D[s]["xyz"]):
            how[s] = "voxel map"                            # the training data are exactly this sub-sample
        else:                                               # pre-processed data in another order: nearest sub-sampled point instead
            _, inv = D[s]["tree"].query(xyz.astype(np.float32), workers=-1); how[s] = "nearest point"
        cm_full += confusion(lab, PRED[s].astype(np.int64)[inv], C)
        cm_orc += confusion(lab, D[s]["label"][inv], C)
        print(f"{s}: {len(lab):,} original points <- {len(D[s]['xyz']):,} sub-sampled ({how[s]})")
        del a, xyz
    return cm_full, cm_orc, how

if CFG["fullres_val"]:
    try:
        cm_full, cm_orc, how = fullres_validation()
        mf = metrics_from_confusion(cm_full, CFG["small"]); mo = metrics_from_confusion(cm_orc, CFG["small"]); n_full = cm_full.sum(1)
        np.save(os.path.join(OUT, "confusion_val_fullres.npy"), cm_full)
        print(f"\n{CFG['run_name']}  |  validation on the ORIGINAL points of {'+'.join(CFG['val'])}  |  best epoch {best_epoch}  |  {'SMOKE TEST - model numbers are meaningless' if SMOKE else 'GPU run'}\n")
        print(f"{'':<26} {'mIoU':>7} {'small mIoU':>11} {'mAcc':>7} {'OA':>7}")
        for name, r in [(f"{CFG['voxel']} points", m), ("original points", mf), (f"oracle ({CFG['voxel']} voxels)", mo)]:
            print(f"{name:<26} {r['mIoU']:>7.2f} {r['small_mIoU']:>11.2f} {r['mAcc']:>7.2f} {r['OA']:>7.2f}")
        print(f"\n{'ID':>2} {'class':<14} {'orig. points':>12} {CFG['voxel'] + ' IoU':>9} {'orig. IoU':>10} {'oracle':>7}")
        for c in range(CFG["num_classes"]):
            if mf["iou"][c] is not None:
                v = f"{m['iou'][c]:>9.2f}" if m["iou"][c] is not None else f"{'-':>9}"
                print(f"{c:>2} {CLASS_NAMES[c]:<14} {n_full[c]:>12,} {v} {mf['iou'][c]:>10.2f} {mo['iou'][c]:>7.2f}{'   <- small component' if c in CFG['small'] else ''}")
        print("\nwhere the original points go:")
        for c in CFG["focus"]: print("  " + readout(cm_full, c))
        final["fullres"] = dict(metrics=mf, oracle=mo, points_per_class=n_full.tolist(), mapping=how,
                                focus_rows={CLASS_NAMES[c]: cm_full[c].tolist() for c in CFG["focus"]},
                                note="focus_rows: confusion-matrix rows, index = predicted class ID")
    except Exception as e:
        import traceback; traceback.print_exc()
        final["fullres_error"] = repr(e)
        print("\n" + "!" * 100 + "\nFULL-RESOLUTION VALIDATION FAILED (the training results above are not affected): " + repr(e) + "\n" + "!" * 100)
    with open(os.path.join(OUT, "final_metrics.json"), "w") as fh: json.dump(final, fh, indent=1)

## Test stations: one label per original point

S170, S171, S180 have no public labels. The files written here (`pred_test/S170labels.npy`, ...) hold one class ID per point, in the order of the points of the test file —
the content of a Codabench submission (`real_labels/gold/`). Check the organisers' example for the expected dtype before the first upload (`docs/dataset_3dses.md`).
A submission should be made only with a run the team has chosen: 4 per day, 50 in total.

In [ ]:
def predict_test_stations():
    od = os.path.join(OUT, "pred_test"); os.makedirs(od, exist_ok=True); info = {}
    for s in CFG["test"]:
        t0 = time.time(); a = np.load(raw_file("test_area", s))
        assert a.ndim == 2 and a.shape[1] >= 7, a.shape
        xyz = a[:, :3] - OFFSET
        keep, inv = voxel_subsample_inv(xyz, VOXEL_M)
        st = dict(xyz=xyz[keep].astype(np.float32), rgb=np.clip(a[keep, 3:6], 0, 255).astype(np.uint8),
                  intensity=a[keep, 6].astype(np.float32), label=np.zeros(len(keep), dtype=np.int64))      # labels unknown: placeholder
        st["tree"] = cKDTree(st["xyz"])
        full = predict_station(st).argmax(1).astype(np.uint8)[inv]
        assert full.shape == (len(a),)
        np.save(os.path.join(od, s + "labels.npy"), full)
        info[s] = dict(points=int(len(full)), points_sub=int(len(keep)), seconds=round(time.time() - t0, 1),
                       predicted_per_class=np.bincount(full, minlength=CFG["num_classes"]).tolist())
        print(f"{s}: {len(full):,} points labelled ({len(keep):,} at {CFG['voxel']}) in {time.time() - t0:.0f} s")
        del a, xyz, st
    return info

if CFG["predict_test"]:
    try:
        tinfo = predict_test_stations(); model.train()
        tot = np.sum([v["predicted_per_class"] for v in tinfo.values()], axis=0)
        print("\npredicted points per class over the three test stations" + ("  (SMOKE TEST - meaningless)" if SMOKE else "") + ":")
        print(", ".join(f"{CLASS_NAMES[c]} {int(tot[c]):,}" for c in range(CFG["num_classes"])))
        final["test_prediction"] = dict(stations=tinfo, files="pred_test/<station>labels.npy, uint8, one class ID per original point in file order")
    except Exception as e:
        import traceback; traceback.print_exc()
        final["test_prediction_error"] = repr(e)
        print("\n" + "!" * 100 + "\nTEST-STATION PREDICTION FAILED (the results above are not affected): " + repr(e) + "\n" + "!" * 100)
    with open(os.path.join(OUT, "final_metrics.json"), "w") as fh: json.dump(final, fh, indent=1)
print("\nsaved:", sorted(os.path.relpath(os.path.join(d, f), OUT) for d, _, fs in os.walk(OUT) for f in fs))

## After the run

Copy into `docs/experiment_log.md` under the run's EXP number: the configuration (run name, seed, intensity), the code commit, the headline numbers
(sub-sampled points **and** original points), the per-class table, the "where the original points go" lines, the training time, and the Kaggle version number.
Download `final_metrics.json` and `log.jsonl` into `results/<run_name>/` in the project folder.

Known simplifications of this baseline (state them in the paper where relevant):
- the checkpoint is chosen by the validation score on the sub-sampled points of the same stations it is then reported on;
- classes that do not occur in S163 + S169 (Railing, Stair, Column) are left out of the validation mIoU;
- the test-station labels come from a model trained on five stations; the final submission should use a model retrained on all seven.